## ex04_ab-test

In [1]:
import pandas as pd
import sqlite3

### создайте подключение к базе данных с помощью библиотеки sqlite3

In [2]:
conn = sqlite3.connect('../data/checking-logs.sqlite')
cur = conn.cursor()

### используя только один запрос для каждой из групп, создайте два фрейма данных: test_results и control_results со столбцами time и avg_diff и только двумя строками

In [3]:
test_results = pd.read_sql("""
            SELECT
                AVG((CAST(strftime('%s', first_commit_ts) AS first) - deadlines) / 3600) AS avg_diff,
                CASE
                   WHEN first_commit_ts <= first_view_ts
                        THEN 'before'
                   ELSE 'after'
                END AS time
            FROM test
            JOIN deadlines ON deadlines.labs = test.labname  
            WHERE labname != 'project1'
            GROUP BY time
            """
            ,conn)
test_results

,avg_diff,time
0,-103.40625,after
1,-60.56250,before


In [4]:
control_results = pd.read_sql("""
            SELECT
                AVG((CAST(strftime('%s', first_commit_ts) AS first) - deadlines) / 3600) AS avg_diff,
                CASE
                   WHEN first_commit_ts <= first_view_ts
                        THEN 'before'
                   ELSE 'after'
                END AS time
            FROM control
            JOIN deadlines ON deadlines.labs = control.labname  
            WHERE labname != 'project1'
            GROUP BY time
            """
            ,conn)
control_results

,avg_diff,time
0,-112.710526,after
1,-99.464286,before


### закройте соединение

In [5]:
conn.close()

### оказалась ли гипотеза верной и влияет ли страница на поведение учащихся?        ДА